In [2]:
import os
import numpy as np
import pandas as pd
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [3]:
import sentence_transformers
import faiss

print("sentence-transformers:", sentence_transformers.__version__)
print("FAISS:", faiss.__version__)

sentence-transformers: 6.0.1
FAISS: 1.15.0


In [4]:
DATA_PATH = "../data/processed/m_absa_selected.csv"

df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Unique reviews:", df["text"].nunique())

Rows: 27582
Unique reviews: 17795


In [5]:
reviews = (
    df[
        [
            "text",
            "language",
            "domain"
        ]
    ]
    .drop_duplicates(subset=["text"])
    .reset_index(drop=True)
)

print("Unique reviews:", len(reviews))
print(reviews.head())

Unique reviews: 17795
                                                text language      domain
0  We have gone for dinner only a few times but t...       en  restaurant
1  Its dark , and cozy . . there is always jazz m...       en  restaurant
2         This place has great indian chinese food .       en  restaurant
3  Not what I would expect for the price and pres...       en  restaurant
4            Finally a reliable Chinese restaurant !       en  restaurant


In [6]:
train_texts = set(
    df[df["split"] == "train"]["text"].unique()
)

test_texts = set(
    df[df["split"] == "test"]["text"].unique()
)

train_reviews = reviews[
    reviews["text"].isin(train_texts)
].reset_index(drop=True)

test_reviews = reviews[
    reviews["text"].isin(test_texts)
].reset_index(drop=True)

print("Train reviews:", len(train_reviews))
print("Test reviews:", len(test_reviews))

Train reviews: 10355
Test reviews: 4958


In [7]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("Embedding model loaded")
print("Device:", embedding_model.device)

Embedding model loaded
Device: cuda:0


In [8]:
from sentence_transformers import SentenceTransformer
import numpy as np
import os

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", embedding_model.device)

Device: cuda:0


In [9]:
train_texts = train_reviews["text"].tolist()

train_embeddings = embedding_model.encode(
    train_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embeddings shape:", train_embeddings.shape)
print("dtype:", train_embeddings.dtype)

Embeddings shape: (10355, 384)
dtype: float32


In [10]:
os.makedirs("../models/embeddings", exist_ok=True)

np.save(
    "../models/embeddings/train_embeddings.npy",
    train_embeddings
)

print("Saved:", train_embeddings.shape)

Saved: (10355, 384)


In [11]:
import faiss

embedding_dim = train_embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)

index.add(
    train_embeddings.astype("float32")
)

print("Embedding dimension:", embedding_dim)
print("Vectors in index:", index.ntotal)

Embedding dimension: 384
Vectors in index: 10355


In [12]:
import os

os.makedirs("../vector_store", exist_ok=True)

faiss.write_index(
    index,
    "../vector_store/index.faiss"
)

train_reviews.to_pickle(
    "../vector_store/metadata.pkl"
)

print("FAISS index saved")
print("Metadata saved")

FAISS index saved
Metadata saved


In [13]:
def semantic_search(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = train_reviews.iloc[indices[0]].copy()
    results["score"] = scores[0]

    return results.reset_index(drop=True)

In [14]:
results = semantic_search(
    "The room was clean and comfortable",
    top_k=5
)

display(results[["text", "language", "domain", "score"]])

,text,language,domain,score
0,Room was very clean .,en,hotel,0.935915
1,Zimmer war sehr sauber.,de,hotel,0.934670
2,Die Zimmer waren sauber.,de,hotel,0.914246
3,Zimmer sauber und einfach eingerichtet.,de,hotel,0.907565
4,Das Zimmer war gemütlich und leicht zugänglich.,de,hotel,0.902819


In [15]:
test_queries = [
    "The room was clean and comfortable",
    "The food was delicious but the service was slow",
    "The laptop battery does not last long",
    "Номер был чистым, но слишком маленьким",
    "Еда была вкусной, но обслуживание медленное",
    "Батарея ноутбука быстро разряжается",
    "Das Zimmer war sauber, aber sehr klein",
    "Das Essen war lecker, aber der Service war langsam",
    "Der Akku des Laptops hält nicht lange"
]

for query in test_queries:
    print("\n" + "=" * 100)
    print("QUERY:", query)

    results = semantic_search(query, top_k=3)

    display(
        results[["text", "language", "domain", "score"]]
    )

,text,language,domain,score
0,Akkulaufzeit: Ich habe den Laptop noch nicht s...,de,laptop,0.877214
1,время работы аккумулятора: я пользуюсь ноутбук...,ru,laptop,0.870011
2,battery life : i haven ' t had the laptop for ...,en,laptop,0.860042


In [16]:
cross_lingual_queries = {
    "EN": "The hotel room was very clean",
    "RU": "Номер в отеле был очень чистым",
    "DE": "Das Hotelzimmer war sehr sauber"
}

for lang, query in cross_lingual_queries.items():
    print("\n" + "=" * 100)
    print(f"{lang}: {query}")

    results = semantic_search(query, top_k=5)

    display(
        results[["text", "language", "domain", "score"]]
    )

,text,language,domain,score
0,Hotel hat viel Charakter und die Zimmer waren ...,de,hotel,0.952393
1,Die Zimmer waren sauber Hotelpersonal freundlich.,de,hotel,0.941017
2,The rooms were clean hotel staff friendly .,en,hotel,0.928731
3,"Die Einrichtung war wirklich modern und schön,...",de,hotel,0.894871
4,Hotel has alot of charater and the rooms were ...,en,hotel,0.893001


In [17]:
retrieval_results = []

for lang, query in cross_lingual_queries.items():

    results = semantic_search(query, top_k=5)

    for rank, row in results.iterrows():
        retrieval_results.append({
            "query_language": lang,
            "query": query,
            "rank": rank + 1,
            "text": row["text"],
            "language": row["language"],
            "domain": row["domain"],
            "score": row["score"]
        })

retrieval_results_df = pd.DataFrame(retrieval_results)

retrieval_results_df.to_csv(
    "../data/processed/retrieval_examples.csv",
    index=False,
    encoding="utf-8"
)

retrieval_results_df.head(15)

,query_language,query,rank,text,language,domain,score
0,EN,The hotel room was very clean,1,Hotel hat viel Charakter und die Zimmer waren ...,de,hotel,0.924704
1,EN,The hotel room was very clean,2,The rooms were clean hotel staff friendly .,en,hotel,0.922473
2,EN,The hotel room was very clean,3,Die Zimmer waren sauber Hotelpersonal freundlich.,de,hotel,0.916726
3,EN,The hotel room was very clean,4,Hotel has alot of charater and the rooms were ...,en,hotel,0.880809
4,EN,The hotel room was very clean,5,"Die Einrichtung war wirklich modern und schön,...",de,hotel,0.876846
5,RU,Номер в отеле был очень чистым,1,"Отель имеет много особенностей, а номера были ...",ru,hotel,0.931051
6,RU,Номер в отеле был очень чистым,2,Hotel hat viel Charakter und die Zimmer waren ...,de,hotel,0.928409
7,RU,Номер в отеле был очень чистым,3,Die Zimmer waren sauber Hotelpersonal freundlich.,de,hotel,0.903461
8,RU,Номер в отеле был очень чистым,4,The rooms were clean hotel staff friendly .,en,hotel,0.902967
9,RU,Номер в отеле был очень чистым,5,"отель был чистым, персонал был отзывчивым, он ...",ru,hotel,0.898210


In [18]:
for lang in cross_lingual_queries:
    scores = retrieval_results_df[
        retrieval_results_df["query_language"] == lang
    ]["score"].tolist()

    print(lang, scores)

EN [0.9247041940689087, 0.9224725365638733, 0.9167258739471436, 0.8808085322380066, 0.8768459558486938]
RU [0.9310509562492371, 0.928409218788147, 0.9034610986709595, 0.9029674530029297, 0.8982099294662476]
DE [0.9523932337760925, 0.941016674041748, 0.928730845451355, 0.8948713541030884, 0.8930009007453918]
